In [1]:
import torch
from torchvision import transforms
import time
import os

from helper import *

torch.set_num_threads(8)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', device)


PyTorch version: 2.12.0+cpu
Device: cpu


In [5]:
IMAGE_SIZE = 32
BATCH_SIZE = 128
NUM_WORKERS = 2

data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616),
    ),
])

train_loader, test_loader, class_names = create_dataloaders(data_transform,BATCH_SIZE, NUM_WORKERS, subset_per_class=4000)



In [ ]:
import torch.nn as nn

class PoolCNN(nn.Module):
    def __init__(self,  pool_layer, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            pool_layer(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            pool_layer(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            pool_layer(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            pool_layer(2),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

In [20]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class BasePool2d(nn.Module):
    def __init__(
        self,
        kernel_size: int = 2,
        stride: int | None = None,
        padding: int = 0,
    ) -> None:
        super().__init__()

        self.kernel_size = kernel_size
        self.stride = stride or kernel_size
        self.padding = padding

    def _extract_patches(
        self,
        x: torch.Tensor,
    ) -> tuple[torch.Tensor, int, int]:
        n, c, h, w = x.shape

        patches = F.unfold(
            x,
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=self.padding,
        )

        k2 = self.kernel_size * self.kernel_size

        patches = patches.view(
            n,
            c,
            k2,
            -1,
        )

        out_h = (
            h + 2 * self.padding - self.kernel_size
        ) // self.stride + 1

        out_w = (
            w + 2 * self.padding - self.kernel_size
        ) // self.stride + 1

        return patches, out_h, out_w

    def _reshape_output(
        self,
        pooled: torch.Tensor,
        out_h: int,
        out_w: int,
    ) -> torch.Tensor:
        
        assert isinstance(pooled, torch.Tensor)
        
        n, c, _ = pooled.shape
        return pooled.view(n, c, out_h, out_w)

class MaxPool2d(BasePool2d):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patches, out_h, out_w = self._extract_patches(x)

        pooled = patches.max(dim=2).values
        
        return self._reshape_output(
            pooled,
            out_h,
            out_w,
        )

class VarianceThresholdPool2d(BasePool2d):
    def __init__(
        self,
        kernel_size=2,
        stride=None,
        padding=0,
        threshold=0.05,
        eps=1e-4,
    ):
        super().__init__(
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )

        self.threshold = threshold
        self.eps = eps

    def forward(self, x):
        patches, out_h, out_w = self._extract_patches(x)

        max_vals = patches.amax(dim=2)
        min_vals = patches.amin(dim=2)

        spread = (max_vals - min_vals).div_(
            min_vals.abs().add_(self.eps)
        )

        pooled = torch.where(
            spread < self.threshold,
            0.0,
            max_vals,
        )

        return self._reshape_output(
            pooled,
            out_h,
            out_w,
        )      

class CVPool2d(BasePool2d):
    def __init__(
        self,
        kernel_size=2,
        stride=None,
        padding=0,
        threshold=0.1,
        eps=1e-4,
    ):
        super().__init__(
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )

        self.threshold = threshold
        self.eps = eps

    def forward(self, x):
        patches, out_h, out_w = self._extract_patches(x)

        mean = patches.mean(dim=2)

        diff = patches - mean.unsqueeze(2)
        var = diff.square().mean(dim=2)

        threshold2 = (
            self.threshold
            * (mean.abs() + self.eps)
        ).square()

        max_vals = patches.amax(dim=2)

        pooled = max_vals.masked_fill(
            var < threshold2,
            0.0,
        )

        return self._reshape_output(
            pooled,
            out_h,
            out_w,
        )
 
        
class SparesCVPool2d(BasePool2d):
    def __init__(
        self,
        kernel_size=2,
        stride=None,
        padding=0,
    ):
        super().__init__(
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        )


    def forward(self, x):
        patches, out_h, out_w = self._extract_patches(x)

        mean = patches.mean(dim=2)
        max_vals = patches.max(dim=2).values

        pooled = torch.where(
            max_vals > mean * 1.15,
            max_vals,
            0,     
        )

        return self._reshape_output(
            pooled,
            out_h,
            out_w,
        )


In [7]:
def run_exper(name, model, train_loader, test_loader, epochs=1):
    log_dir = os.path.join('runs/pooling_layer', f'Pooling_layer_cifar10_{name}_{int(time.time())}')
    os.makedirs(log_dir, exist_ok=True)
    set_seed(42)  # re-set seed for deterministic execution within the thread
    _, acc = run_training(model, train_loader, test_loader, log_dir, device, IMAGE_SIZE, epochs)
    return name, acc


In [ ]:

custom_experiments = [    
    ("varianceMax", PoolCNN(CVPool2d), train_loader, test_loader),
]


for exp in custom_experiments:
    name, acc = run_exper(*exp,15)

In [9]:
#  To display the tensorboard  , download the tensorboard library and run the code below

#tensorboard --logdir runs/pooling_layer --bind_all 